修改embedding model: bert-base-chinese -> 'intfloat/multilingual-e5-base'

Grid Search Results (multilingual-e5-base)
=========================================
Val best overall loss: 0.4247, Val AUC: 0.7863


Best Hyperparameters:
  DKT_HIDDEN_DIM   : 1024
  DKT_NUM_LAYERS   : 2
  DKT_DROPOUT      : 0.2
  BATCH_SIZE       : 32
  LEARNING_RATE    : 0.001


Score:
Public score: 0.7460

EMBED_DIM = 768  # multilingual-e5-base also outputs 768
MAX_SEQ_LEN = 100 
EPOCHS = 50
 
比較好一點點!!


In [ ]:
import os
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel 
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import roc_auc_score, f1_score
from tqdm import tqdm
import pickle


# --- Configuration ---
EMBED_DIM = 768  # multilingual-e5-base also outputs 768
MAX_SEQ_LEN = 100
EPOCHS = 50
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 嵌入模型名稱 (Changed to multilingual-e5-base)
EMBED_MODEL_NAME = 'intfloat/multilingual-e5-base'

# 設定檔案路徑
base_data_path = './data/'
train_file = os.path.join(base_data_path, 'train.csv')
test_file = os.path.join(base_data_path, 'test.csv')
questions_file = os.path.join(base_data_path, 'questions.json')
concepts_file = os.path.join(base_data_path, 'concept.json')

# 儲存 embedding 的路徑 (Updated filenames for the new model)
embedding_path = './embeddings/'
q_embed_file = os.path.join(embedding_path, 'question_embeddings_e5_base.pt')
c_embed_file = os.path.join(embedding_path, 'concept_embeddings_e5_base.pt') 
processed_train_data_file = os.path.join(embedding_path , 'processed_train_data_e5_base.pkl') 
processed_test_data_file = os.path.join(embedding_path, 'processed_test_data_e5_base.pkl') 

result_path = './result/'
submission_file_path = os.path.join(result_path, 'submission_e5_base.csv') 

os.makedirs(base_data_path, exist_ok=True) 
os.makedirs(embedding_path, exist_ok=True)
os.makedirs(result_path, exist_ok=True) 

# --- Grid Search 參數設定 ---
# param_grid = {
#     'DKT_HIDDEN_DIM': [128, 256, 512],
#     'DKT_NUM_LAYERS': [2, 3, 4],
#     'DKT_DROPOUT': [0.2, 0.3],
#     'LEARNING_RATE': [1e-3, 1e-4], 
#     'BATCH_SIZE': [32, 64, 128] 
# }

# best param
param_grid = {
    'DKT_HIDDEN_DIM': [1024], 
    'DKT_NUM_LAYERS': [2], 
    'DKT_DROPOUT': [0.3],
    'LEARNING_RATE': [1e-3], 
    'BATCH_SIZE': [32] 
}

grid = ParameterGrid(param_grid)

EARLY_STOPPING_PATIENCE = 5


c:\Users\s1092\Desktop\DataMining_final\data\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 資料載入與初步處理

In [2]:

# --- 資料載入與初步處理 ---
def load_data():
    print("Loading data...")
    try:
        train_df = pd.read_csv(train_file)
        test_df = pd.read_csv(test_file)
    except FileNotFoundError as e:
        print(f"Error: {e}. Make sure data files are in '{base_data_path}'") 
        print("Please create a './data/' directory and place train.csv, test.csv, questions.json, and concept.json there.") 
        return None, None, None, None

    with open(questions_file, 'r', encoding='utf-8') as f:
        questions_data = json.load(f)
    with open(concepts_file, 'r', encoding='utf-8') as f:
        concepts_data = json.load(f)

    print("Data loaded.")
    return train_df, test_df, questions_data, concepts_data

def preprocess_concept_id(df):
    df['concept_id'] = df['concept_id'].astype(str)
    df['concept_id_list'] = df['concept_id'].apply(lambda x: x.split('_') if pd.notnull(x) else []) 
    return df


# Embeddings 生成

In [3]:

# --- Embeddings 生成 ---
def get_embeddings(text_data, tokenizer, model, device, is_query=False): # Added is_query, though not strictly used for differentiation here yet
    """使用 BERT 模型為文本生成 embedding"""
    # Prepend "query: " or "passage: " for E5 models
    # For this script, question content and concepts are more like passages.
    prefixed_text_data = [f"passage: {text}" for text in text_data]

    inputs = tokenizer(prefixed_text_data, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {key: val.to(device) for key, val in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1) # Mean pooling
    return embeddings.cpu()

def generate_question_embeddings(questions_data, tokenizer, embed_model, device):
    print("Generating question embeddings...")
    if os.path.exists(q_embed_file):
        print(f"Loading cached question embeddings from {q_embed_file}")
        return torch.load(q_embed_file)

    question_embeddings = {}
    all_question_ids = [] 
    all_texts_to_embed = []

    for q_id, q_info in tqdm(questions_data.items()):
        content = q_info.get('content', '')
        answer = q_info.get('answer', '') 
        analysis = q_info.get('analysis', '') 
        q_type = q_info.get('type', '')
        kc_routes_list = q_info.get('kc_routes', []) 
        kc_routes_str = " #KC_SEP# ".join(kc_routes_list) if isinstance(kc_routes_list, list) else str(kc_routes_list)
        options_str = str(q_info.get('options', ''))
        full_text = f"內容: {content} 答案: {answer} 解析: {analysis} 類型: {q_type} 知識點路徑: {kc_routes_str} 選項: {options_str}" 
        all_question_ids.append(q_id) 
        all_texts_to_embed.append(full_text)

    batch_size_embed = 64
    num_batches = (len(all_texts_to_embed) + batch_size_embed - 1) // batch_size_embed 
    temp_embeddings_list = [] 

    for i in tqdm(range(num_batches)):
        batch_texts = all_texts_to_embed[i*batch_size_embed : (i+1)*batch_size_embed]
        if batch_texts: 
             batch_embeddings = get_embeddings(batch_texts, tokenizer, embed_model, device) 
             temp_embeddings_list.append(batch_embeddings) 

    if not temp_embeddings_list: 
        print("Warning: No question texts found to embed.") 
        final_embeddings_tensor = torch.empty(0, EMBED_DIM) 
    else:
        final_embeddings_tensor = torch.cat(temp_embeddings_list, dim=0) 

    for i, q_id in enumerate(all_question_ids):
        question_embeddings[q_id] = final_embeddings_tensor[i]

    torch.save(question_embeddings, q_embed_file) 
    print(f"Question embeddings saved to {q_embed_file}") 
    return question_embeddings

def generate_concept_embeddings(concepts_data, tokenizer, embed_model, device):
    print("Generating concept embeddings...")
    if os.path.exists(c_embed_file):
        print(f"Loading cached concept embeddings from {c_embed_file}") 
        return torch.load(c_embed_file) 

    concept_embeddings = {}
    all_concept_ids = [] 
    all_texts_to_embed = [] 

    for c_id, c_name in tqdm(concepts_data.items()):
        all_concept_ids.append(c_id)
        all_texts_to_embed.append(c_name if isinstance(c_name, str) else str(c_name))

    batch_size_embed = 256 
    num_batches = (len(all_texts_to_embed) + batch_size_embed - 1) // batch_size_embed
    temp_embeddings_list = [] 

    for i in tqdm(range(num_batches)):
        batch_texts = all_texts_to_embed[i*batch_size_embed : (i+1)*batch_size_embed]
        if batch_texts: 
            batch_embeddings = get_embeddings(batch_texts, tokenizer, embed_model, device) 
            temp_embeddings_list.append(batch_embeddings) 

    if not temp_embeddings_list:
        print("Warning: No concept texts found to embed.")
        final_embeddings_tensor = torch.empty(0, EMBED_DIM) 
    else:
        final_embeddings_tensor = torch.cat(temp_embeddings_list, dim=0) 

    for i, c_id in enumerate(all_concept_ids): 
        concept_embeddings[c_id] = final_embeddings_tensor[i] 

    torch.save(concept_embeddings, c_embed_file) 
    print(f"Concept embeddings saved to {c_embed_file}") 
    return concept_embeddings


# 特徵工程與序列構建 

In [4]:

# --- 特徵工程與序列構建 ---
def create_features(df, q_embeddings, c_embeddings):
    print("Creating features...")
    df['question_id_str'] = df['question_id'].astype(str) 
    zero_q_emb = torch.zeros(EMBED_DIM) 
    df['q_emb'] = df['question_id_str'].apply(lambda x: q_embeddings.get(x, zero_q_emb)) 
    zero_c_emb = torch.zeros(EMBED_DIM) 

    def get_avg_concept_emb(c_id_list):
        if not c_id_list:
            return zero_c_emb
        embs = [c_embeddings.get(str(c_id), zero_c_emb) for c_id in c_id_list] 
        valid_embs = [emb for emb in embs if torch.is_tensor(emb) and emb.shape[0] == EMBED_DIM] 
        if not valid_embs:
            return zero_c_emb 
        return torch.stack(valid_embs).mean(dim=0) 

    df['c_emb'] = df['concept_id_list'].apply(get_avg_concept_emb) 
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms') 
    df = df.sort_values(by=['uid', 'timestamp']) 
    df['time_delta'] = df.groupby('uid')['timestamp'].diff().dt.total_seconds().fillna(0) 
    df['time_delta'] = np.clip(df['time_delta'], 0, df['time_delta'].quantile(0.99)) 

    if 'response' in df.columns:
        df['prev_response'] = df.groupby('uid')['response'].shift(1).fillna(0.5) 
    else: # For test_df
        df['prev_response'] = 0.5 

    df['attempt_count'] = df.groupby('uid').cumcount() + 1 
    return df

def create_sequences(df, max_seq_len, is_train=True):
    print(f"Creating sequences (max_seq_len={max_seq_len})...")
    user_groups = df.groupby('uid')
    feature_dim = EMBED_DIM * 2 + 3 
    all_sequences = []

    for uid, group in tqdm(user_groups):
        q_emb_seq = torch.stack(group['q_emb'].tolist())
        c_emb_seq = torch.stack(group['c_emb'].tolist())
        time_delta_scaled = (group['time_delta'] - group['time_delta'].mean()) / (group['time_delta'].std() + 1e-6)
        attempt_count_scaled = (group['attempt_count'] - group['attempt_count'].mean()) / (group['attempt_count'].std() + 1e-6) 
        time_delta_seq = torch.tensor(time_delta_scaled.fillna(0).values, dtype=torch.float).unsqueeze(1)
        prev_response_seq = torch.tensor(group['prev_response'].values, dtype=torch.float).unsqueeze(1)
        attempt_count_seq = torch.tensor(attempt_count_scaled.fillna(0).values, dtype=torch.float).unsqueeze(1) 
        features_seq = torch.cat((q_emb_seq, c_emb_seq, time_delta_seq, prev_response_seq, attempt_count_seq), dim=1) 

        if is_train:
            responses_seq = torch.tensor(group['response'].values, dtype=torch.float) 
        else: # For test data, response is what we want to predict
            responses_seq = torch.zeros(len(group), dtype=torch.float) 

        seq_len = len(group)

        if seq_len >= max_seq_len:
            current_features = features_seq[-max_seq_len:]
            if is_train:
                current_responses = responses_seq[-max_seq_len:]
            else: # test
                current_responses = responses_seq[-max_seq_len:]
            current_mask = torch.ones(max_seq_len, dtype=torch.bool)
        else: # seq_len < max_seq_len (padding)
            pad_len = max_seq_len - seq_len 
            feature_padding = torch.zeros(pad_len, feature_dim)
            current_features = torch.cat((features_seq, feature_padding), dim=0) 

            if is_train:
                response_padding = torch.zeros(pad_len) # Pad with 0 for response, will be masked out 
                current_responses = torch.cat((responses_seq, response_padding), dim=0) 
            else: # test
                current_responses = torch.cat((responses_seq, torch.zeros(pad_len)), dim=0) 

            current_mask = torch.cat((torch.ones(seq_len, dtype=torch.bool), torch.zeros(pad_len, dtype=torch.bool)), dim=0) 

        all_sequences.append({
            'uid': uid, 
            'features': current_features, # (max_seq_len, feature_dim) 
            'responses': current_responses, # (max_seq_len) 
            'mask': current_mask, # (max_seq_len) -> indicates actual interactions
            'original_indices': group.index.tolist() # Store original indices for mapping predictions back for test set 
        })

    return all_sequences, feature_dim 


# PyTorch Dataset and DataLoader

In [5]:

# --- PyTorch Dataset and DataLoader ---
class KnowledgeTracingDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return {
            'features': self.sequences[idx]['features'], 
            'responses': self.sequences[idx]['responses'],
            'mask': self.sequences[idx]['mask'],
            'uid': self.sequences[idx]['uid'] 
        }


# DKTModel

In [6]:

# --- DKT Model ---
class DKTModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, dropout):
        super(DKTModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, dropout=dropout if num_layers > 1 else 0) 
        self.fc = nn.Linear(hidden_dim, 1) # Output one value (probability) 
        self.sigmoid = nn.Sigmoid() 
        
    def forward(self, x, h_0=None, c_0=None):
        if h_0 is not None and c_0 is not None:
            out, (hn, cn) = self.lstm(x, (h_0, c_0)) 
        else:
            out, (hn, cn) = self.lstm(x) 
        predictions = self.fc(out) # (batch_size, seq_len, 1)
        predictions = self.sigmoid(predictions.squeeze(-1)) # (batch_size, seq_len) 
        return predictions, (hn, cn)


# Training and Evaluation

In [7]:

# --- Training and Evaluation ---
def train_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    all_preds = []
    all_targets = []
    all_masks = [] 

    for batch in tqdm(dataloader, desc="Training"):
        features = batch['features'].to(device)
        responses = batch['responses'].to(device)
        mask = batch['mask'].to(device) # Mask for valid timesteps 
        optimizer.zero_grad()
        preds, _ = model(features) 
        masked_preds = preds[mask] 
        masked_responses = responses[mask] 
        
        if masked_preds.nelement() == 0: # Skip if batch has no valid interactions (e.g. all padding)
            continue

        loss = criterion(masked_preds, masked_responses) 
        loss.backward() 
        optimizer.step()
        total_loss += loss.item() * masked_preds.nelement() # Weighted by number of valid elements
        all_preds.append(masked_preds.detach().cpu().numpy())
        all_targets.append(masked_responses.detach().cpu().numpy()) 

    if not all_preds: # Handle case where no predictions were made
        return 0.0, 0.0, 0.0 

    avg_loss = total_loss / np.concatenate(all_targets).shape[0] if np.concatenate(all_targets).shape[0] > 0 else 0 
    all_preds_flat = np.concatenate(all_preds) 
    all_targets_flat = np.concatenate(all_targets) 
    auc = roc_auc_score(all_targets_flat, all_preds_flat)
    f1 = f1_score(all_targets_flat, (all_preds_flat > 0.5).astype(int)) 
    return avg_loss, auc, f1

def evaluate_epoch(model, dataloader, criterion, device):
    model.eval() 
    total_loss = 0
    all_preds = []
    all_targets = [] 

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            features = batch['features'].to(device)
            responses = batch['responses'].to(device) 
            mask = batch['mask'].to(device) 
            preds, _ = model(features) 
            masked_preds = preds[mask]
            masked_responses = responses[mask] 

            if masked_preds.nelement() == 0: 
                continue

            loss = criterion(masked_preds, masked_responses) 
            total_loss += loss.item() * masked_preds.nelement() 
            all_preds.append(masked_preds.cpu().numpy()) 
            all_targets.append(masked_responses.cpu().numpy()) 

    if not all_preds: # Handle case where no predictions were made
        return 0.0, 0.0, 0.0 

    avg_loss = total_loss / np.concatenate(all_targets).shape[0] if np.concatenate(all_targets).shape[0] > 0 else 0 
    all_preds_flat = np.concatenate(all_preds)
    all_targets_flat = np.concatenate(all_targets)
    auc = roc_auc_score(all_targets_flat, all_preds_flat) 
    f1 = f1_score(all_targets_flat, (all_preds_flat > 0.5).astype(int)) 
    return avg_loss, auc, f1


# Prediction for Test Set

In [8]:

# --- Prediction for Test Set ---
def predict_test_set(model, test_sequences_data, device, user_last_response_train=None): 
    """
    Generates predictions for the test set. 
    If a student has multiple test interactions, the prev_response for the current test interaction 
    should be the model's prediction for the previous test interaction. 
    """
    model.eval()
    predictions_map = {} # Stores original_index: prediction 
    print("Predicting on test set...")

    for seq_data in tqdm(test_sequences_data, desc="Predicting Test"):
        uid = seq_data['uid']
        features_full_seq = seq_data['features'].unsqueeze(0).to(device) # (1, max_seq_len, feature_dim) 
        mask_full_seq = seq_data['mask'].unsqueeze(0).to(device) # (1, max_seq_len) 
        original_indices_user = seq_data['original_indices'] # List of original df indices for this user's seq 
        actual_interaction_indices_in_seq = torch.where(mask_full_seq.squeeze(0))[0] 

        if len(actual_interaction_indices_in_seq) == 0:
            continue # No actual interactions for this user in this sequence 

        h_prev, c_prev = None, None 
        current_features_for_lstm = features_full_seq.clone() # Make a mutable copy 
        prev_response_feature_idx = EMBED_DIM * 2 + 1 # 0-indexed 
        last_known_response = 0.5 # Default if no prior 
        if user_last_response_train and uid in user_last_response_train:
            last_known_response = user_last_response_train[uid] 

        if len(actual_interaction_indices_in_seq) > 0:
            first_actual_idx_in_seq = actual_interaction_indices_in_seq[0] 
            current_features_for_lstm[0, first_actual_idx_in_seq, prev_response_feature_idx] = last_known_response

        output_preds_seq, _ = model(current_features_for_lstm, h_prev, c_prev) # (1, max_seq_len) 

        for i, seq_idx in enumerate(actual_interaction_indices_in_seq):
            original_df_idx = original_indices_user[seq_idx.item()] # Get original index from test_df 
            prediction = output_preds_seq[0, seq_idx.item()].item() 
            predictions_map[original_df_idx] = prediction 
            # Update prev_response for the next interaction in the same sequence *if* it exists
            if i + 1 < len(actual_interaction_indices_in_seq):
                next_actual_idx_in_seq = actual_interaction_indices_in_seq[i+1]
                current_features_for_lstm[0, next_actual_idx_in_seq, prev_response_feature_idx] = prediction
                # Re-predict from this point if we want true sequential dependency for test set.
                # However, the original code processes the whole sequence at once.
                # For simplicity and consistency with original logic of one pass per user sequence:
                # The current structure does one pass per sequence. If strict sequential update of prev_response
                # within a test sequence is desired for LSTM state propagation, the loop would need refactoring
                # to predict one step at a time. The current code uses the batch prediction `output_preds_seq`
                # and updates the input features for the *next* user (if any) or based on training.
                # The current code takes the prediction for seq_idx and then for the next seq_idx_plus_1,
                # the prev_response feature in `current_features_for_lstm` would be based on original test data or `last_known_response`
                # not the `prediction` from `seq_idx` unless explicitly handled in a step-by-step manner.
                # The provided code structure implies that `prev_response` for test items within a sequence is derived
                # from the initial test data (which is 0.5 or imputed).
                # The text "the prev_response for the current test interaction should be the model's prediction for the previous test interaction."
                # suggests a step-by-step prediction within the sequence for multi-interaction users in test.
                # This part is tricky with batch LSTM processing.
                # The current code does:
                # 1. Load features_full_seq (prev_response is from test_df_featured, usually 0.5)
                # 2. Update the *first* actual interaction's prev_response with last_known_response from training.
                # 3. Get all output_preds_seq in one go.
                # This means `output_preds_seq` for item j is based on prev_response of item j-1 from initial data, not prediction of item j-1.
                # To implement the "model's prediction for the previous test interaction":
                # We would need to iterate one by one for actual interactions if we want to use the model's *own* previous prediction
                # as input to the *next* step within the same sequence.
                # Given the current overall structure, a full step-by-step inference within `predict_test_set`
                # would be a significant change. The current method uses the pre-calculated `prev_response` from `create_features`
                # for all but the very first step of a sequence (which might get `user_last_response_train`).

    return predictions_map



# Main Execution

In [9]:

def save_best_info(best_auc, best_loss, best_hyperparameters):
    file_path = os.path.join(result_path, "best_info_e5_base.txt") # Updated filename
    with open(file_path, "w", encoding="utf-8") as f: 
        f.write("Grid Search Results (multilingual-e5-base)\n")
        f.write("=========================================\n")
        f.write(f"Val best overall loss: {best_loss:.4f}, Val AUC: {best_auc:.4f}\n\n")

        f.write("Best Hyperparameters:\n")
        f.write(f"  DKT_HIDDEN_DIM   : {best_hyperparameters.get('DKT_HIDDEN_DIM', 'N/A')}\n")
        f.write(f"  DKT_NUM_LAYERS   : {best_hyperparameters.get('DKT_NUM_LAYERS', 'N/A')}\n")
        f.write(f"  DKT_DROPOUT      : {best_hyperparameters.get('DKT_DROPOUT', 'N/A')}\n")
        f.write(f"  BATCH_SIZE       : {best_hyperparameters.get('BATCH_SIZE', 'N/A')}\n")
        f.write(f"  LEARNING_RATE    : {best_hyperparameters.get('LEARNING_RATE', 'N/A')}\n")

# --- Main Execution ---
if __name__ == '__main__':
    print(f"Using device: {DEVICE}") 
    print(f"Loading HuggingFace tokenizer and model: {EMBED_MODEL_NAME}...") 
    try:
        tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME) 
        embed_model = AutoModel.from_pretrained(EMBED_MODEL_NAME).to(DEVICE) 
        embed_model.eval() # Set to eval mode as we are only using it for feature extraction
    except Exception as e:
        print(f"Error loading BERT model: {e}. Please ensure you have an internet connection or the model is cached.") 
        exit() 
    print("HuggingFace model loaded.") 

    train_df_raw, test_df_raw, questions_data, concepts_data = load_data()

    if train_df_raw is None:
        exit()

    train_df_raw = preprocess_concept_id(train_df_raw.copy()) 
    test_df_raw = preprocess_concept_id(test_df_raw.copy()) 

    questions_data_str_keys = {str(k): v for k, v in questions_data.items()} 
    concepts_data_str_keys = {str(k): v for k, v in concepts_data.items()} 
    q_embeddings = generate_question_embeddings(questions_data_str_keys, tokenizer, embed_model, DEVICE) 
    c_embeddings = generate_concept_embeddings(concepts_data_str_keys, tokenizer, embed_model, DEVICE) 

    if os.path.exists(processed_train_data_file):
        print(f"Loading processed train data from {processed_train_data_file}...") 
        with open(processed_train_data_file, 'rb') as f:
            train_sequences, feature_dim = pickle.load(f) 
    else:
        print("Processing training data...") 
        train_df_featured = create_features(train_df_raw.copy(), q_embeddings, c_embeddings) 
        user_last_response_train = train_df_featured.groupby('uid')['response'].last().to_dict()
        train_sequences, feature_dim = create_sequences(train_df_featured, MAX_SEQ_LEN, is_train=True) 
        with open(processed_train_data_file, 'wb') as f:
            pickle.dump((train_sequences, feature_dim), f) 
        print(f"Processed train data saved to {processed_train_data_file}") 
        
    if os.path.exists(processed_test_data_file):
        print(f"Loading processed test data from {processed_test_data_file}...") 
        with open(processed_test_data_file, 'rb') as f:
            test_sequences, _ = pickle.load(f) # feature_dim should be same 
    else:
        print("Processing test data...") 
        # Need user_last_response_train for create_features if it's used for test_df 'prev_response' initial fill,
        # but create_features for test_df fills 'prev_response' with 0.5 by default.
        # It's mainly predict_test_set that might use user_last_response_train.
        if 'user_last_response_train' not in locals(): # Ensure it exists if needed by logic below
             _train_df_temp_for_last_response = create_features(train_df_raw.copy(), q_embeddings, c_embeddings)
             user_last_response_train = _train_df_temp_for_last_response.groupby('uid')['response'].last().to_dict()

        test_df_featured = create_features(test_df_raw.copy(), q_embeddings, c_embeddings) 
        test_sequences, _ = create_sequences(test_df_featured, MAX_SEQ_LEN, is_train=False) 
        with open(processed_test_data_file, 'wb') as f:
            pickle.dump((test_sequences, feature_dim), f)
        print(f"Processed test data saved to {processed_test_data_file}")

    print(f"Feature dimension: {feature_dim}")
    train_seq_list, val_seq_list = train_test_split(train_sequences, test_size=0.2, random_state=42) 
    train_dataset = KnowledgeTracingDataset(train_seq_list) 
    val_dataset = KnowledgeTracingDataset(val_seq_list) 

    print("Starting DKT model training with Grid Search using multilingual-e5-base embeddings...")
    best_overall_auc = 0.0
    best_overall_loss = float('inf') # Initialize with infinity for loss
    best_params = None
    best_model_path = os.path.join(result_path, "dkt_model_best_e5_base.pt") # Updated

    for params in grid:
        print(f"\n--- Testing Parameters: {params} ---")
        current_hidden_dim = params['DKT_HIDDEN_DIM']
        current_num_layers = params['DKT_NUM_LAYERS']
        current_dropout = params['DKT_DROPOUT']
        current_lr = params['LEARNING_RATE']
        current_batch_size = params['BATCH_SIZE']

        train_dataloader = DataLoader(train_dataset, batch_size=current_batch_size, shuffle=True)
        val_dataloader = DataLoader(val_dataset, batch_size=current_batch_size, shuffle=False) 

        dkt_model = DKTModel(input_dim=feature_dim,
                             hidden_dim=current_hidden_dim,
                             num_layers=current_num_layers,
                             dropout=current_dropout).to(DEVICE)

        optimizer = torch.optim.Adam(dkt_model.parameters(), lr=current_lr)
        criterion = nn.BCELoss() 

        best_val_auc_for_this_run = 0
        epochs_without_improvement = 0

        for epoch in range(EPOCHS):
            train_loss, train_auc, train_f1 = train_epoch(dkt_model, train_dataloader, optimizer, criterion, DEVICE) 
            val_loss, val_auc, val_f1 = evaluate_epoch(dkt_model, val_dataloader, criterion, DEVICE) 

            print(f"Epoch {epoch+1}/{EPOCHS} | Params: {params}") 
            print(f"  Train Loss: {train_loss:.4f}, Train AUC: {train_auc:.4f}, Train F1: {train_f1:.4f}") 
            print(f"  Val Loss  : {val_loss:.4f}, Val AUC  : {val_auc:.4f}, Val F1: {val_f1:.4f}") 

            if val_auc > best_val_auc_for_this_run:
                best_val_auc_for_this_run = val_auc 
                epochs_without_improvement = 0 
                if val_auc > best_overall_auc: # Check if it's the best overall based on AUC
                    best_overall_auc = val_auc
                    best_overall_loss = val_loss # Also save loss for this best AUC model
                    best_params = params
                    torch.save(dkt_model.state_dict(), best_model_path)
                    print(f"  *** New Best Overall Model Saved to {best_model_path} (Val AUC: {best_overall_auc:.4f}) ***") 
                # If AUC is the same, prefer lower loss (optional refinement)
                elif val_auc == best_overall_auc and val_loss < best_overall_loss :
                    best_overall_loss = val_loss
                    best_params = params
                    torch.save(dkt_model.state_dict(), best_model_path)
                    print(f"  *** New Best Overall Model (Lower Loss) Saved to {best_model_path} (Val AUC: {best_overall_auc:.4f}, Val Loss: {best_overall_loss:.4f}) ***")

            else:
                epochs_without_improvement += 1 

            if epochs_without_improvement >= EARLY_STOPPING_PATIENCE: 
                print(f"  Early stopping triggered after {epoch+1} epochs due to no improvement in Val AUC.")
                break

    print("\n--- Grid Search Finished ---")
    print(f"Best validation AUC achieved: {best_overall_auc:.4f}") 
    print(f"Best Parameters found: {best_params}")
    if best_params: # Only save if a best model was found
        save_best_info(best_overall_auc, best_overall_loss, best_params)
    else:
        print("No best model found during grid search (possibly no training runs completed or improved).")


    if best_params is None:
        print("No best parameters found from Grid Search. Exiting before final prediction.")
        exit()

    print(f"Loading best model from {best_model_path} for final predictions...") 
    best_dkt_model = DKTModel(input_dim=feature_dim,
                              hidden_dim=best_params['DKT_HIDDEN_DIM'],
                              num_layers=best_params['DKT_NUM_LAYERS'], 
                              dropout=best_params['DKT_DROPOUT']).to(DEVICE) 
    best_dkt_model.load_state_dict(torch.load(best_model_path)) 

    # Regenerate user_last_response_train if not available, using the new embeddings
    if 'user_last_response_train' not in locals() or user_last_response_train is None: 
        print("Recomputing user_last_response_train for test predictions...")
        # Ensure q_embeddings and c_embeddings are from the new model
        _train_df_temp = create_features(train_df_raw.copy(), q_embeddings, c_embeddings) # Recompute with correct embeddings 
        user_last_response_train = _train_df_temp.groupby('uid')['response'].last().to_dict() 
    elif not user_last_response_train: # If it's an empty dict
        print("Warning: user_last_response_train is empty. Test predictions might be less accurate for initial steps.") 


    test_predictions_map = predict_test_set(best_dkt_model, test_sequences, DEVICE,
                                            user_last_response_train=user_last_response_train) 

    predictions_series = pd.Series(test_predictions_map) 
    predictions_series.name = 'response' 
    submission_df = test_df_raw.copy() 
    # Ensure 'uid' is of a consistent type if it's part of the index or merge key later
    # submission_df['uid'] = submission_df['uid'].astype(int) # Or str, depending on original_indices_user

    # Align indices for merging. test_predictions_map keys are original df indices.
    submission_df = submission_df.merge(predictions_series, left_index=True, right_index=True, how='left') 
    submission_df['response'] = submission_df['response'].fillna(0.5)

    # Select only required columns for submission: 'uid' and 'response' from the *original* test_df_raw structure
    # The predictions are mapped by original index, so we need to ensure the 'uid' corresponds to those.
    # The problem implies submission needs uid and the predicted response.
    # The test_df_raw already has the 'uid' in the correct order corresponding to its original indices.
    final_submission_df = submission_df[['uid', 'response']] 

    final_submission_df.to_csv(submission_file_path, index=False) 
    print(f"Submission file created at: {submission_file_path}") 
    print(f"Submission file has {len(final_submission_df)} rows.") 
    print("Columns:", final_submission_df.columns.tolist())
    print(final_submission_df.head()) 

Using device: cuda
Loading HuggingFace tokenizer and model: intfloat/multilingual-e5-base...
HuggingFace model loaded.
Loading data...
Data loaded.
Generating question embeddings...


100%|██████████| 120/120 [01:10<00:00,  1.70it/s]


Question embeddings saved to ./embeddings/question_embeddings_e5_base.pt
Generating concept embeddings...


100%|██████████| 5/5 [00:00<00:00, 10.33it/s]


Concept embeddings saved to ./embeddings/concept_embeddings_e5_base.pt
Processing training data...
Creating features...
Creating sequences (max_seq_len=100)...


...

Epoch 45/50 | Params: {'BATCH_SIZE': 32, 'DKT_DROPOUT': 0.3, 'DKT_HIDDEN_DIM': 1024, 'DKT_NUM_LAYERS': 2, 'LEARNING_RATE': 0.001}
  Train Loss: 0.4110, Train AUC: 0.8055, Train F1: 0.8909
  Val Loss  : 0.4272, Val AUC  : 0.7825, Val F1: 0.8877


Evaluating: 100%|██████████| 23/23 [00:00<00:00, 52.86it/s]


Epoch 46/50 | Params: {'BATCH_SIZE': 32, 'DKT_DROPOUT': 0.3, 'DKT_HIDDEN_DIM': 1024, 'DKT_NUM_LAYERS': 2, 'LEARNING_RATE': 0.001}
  Train Loss: 0.4093, Train AUC: 0.8073, Train F1: 0.8914
  Val Loss  : 0.4266, Val AUC  : 0.7835, Val F1: 0.8882
  Early stopping triggered after 46 epochs due to no improvement in Val AUC.

--- Grid Search Finished ---
Best validation AUC achieved: 0.7843
Best Parameters found: {'BATCH_SIZE': 32, 'DKT_DROPOUT': 0.3, 'DKT_HIDDEN_DIM': 1024, 'DKT_NUM_LAYERS': 2, 'LEARNING_RATE': 0.001}
Loading best model from ./result/dkt_model_best_e5_base.pt for final predictions...
Predicting on test set...


Predicting Test: 100%|██████████| 3613/3613 [00:51<00:00, 69.95it/s]

Submission file created at: ./result/submission_e5_base.csv
Submission file has 3613 rows.
Columns: ['uid', 'response']
    uid  response
0  8572  0.472436
1   179  0.891018
2  5664  0.694008
3  3587  0.852512
4  6711  0.860877
